# Fermetures

## Temps écoulé depuis la création d'une fermeture

Écrivez une fonction `fabrique_chrono()` qui renvoie une fermeture : à chaque appel, celle-ci renvoie le temps écoulé, en secondes, depuis sa création. Mesurez ce temps avec `time.perf_counter`, faite pour les durées : contrairement à `time.time`, elle ne suit pas les réglages de l'horloge du système.

Créez `chrono = fabrique_chrono()`, attendez avec `sleep(1)`, affichez `chrono()`, puis recommencez : vous devez obtenir environ 1, puis environ 2.

**Documentation utile**

- [`time.perf_counter`](https://docs.python.org/fr/3/library/time.html#time.perf_counter)
- [`time.sleep`](https://docs.python.org/fr/3/library/time.html#time.sleep)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Dans `fabrique_chrono`, gardez `perf_counter()` dans une variable `debut`, puis définissez et renvoyez une fonction interne qui calcule la différence avec un nouvel appel de `perf_counter()`. Elle ne fait que lire `debut` : il n'y a rien à déclarer.

```python
from collections.abc import Callable
from time import perf_counter, sleep


def fabrique_chrono() -> Callable[[], float]:
  debut = perf_counter()

  def chrono() -> float:
    return ...

  return chrono
```

### Solution

In [ ]:
from collections.abc import Callable
from time import perf_counter, sleep


def fabrique_chrono() -> Callable[[], float]:
  debut = perf_counter()

  def chrono() -> float:
    return perf_counter() - debut

  return chrono


chrono = fabrique_chrono()
sleep(1)
print(chrono())
sleep(1)
print(chrono())

On peut récupérer les noms des variables capturées par la fermeture dans `__code__.co_freevars`, et leurs valeurs dans `__closure__[index].cell_contents`.

In [ ]:
print(chrono.__code__.co_freevars, chrono.__closure__[0].cell_contents)

## Un compteur par fermeture

Écrivez une fonction `fabrique_compteur(pas)` qui renvoie une fermeture : à chaque appel, celle-ci ajoute `pas` à son compteur, parti de 0, et renvoie la nouvelle valeur.

Créez `unites = fabrique_compteur(1)` et `dizaines = fabrique_compteur(10)`, et vérifiez que les deux compteurs évoluent séparément : `unites()`, `unites()`, `dizaines()` et `unites()`, appelés dans cet ordre, doivent renvoyer 1, 2, 10 et 3.

In [ ]:
# Votre code ici

### Conseils supplémentaires

La fermeture réaffecte le compteur : sans déclaration `nonlocal`, `valeur += pas` lève une `UnboundLocalError`. Chaque appel de `fabrique_compteur` crée ses propres variables, d'où deux compteurs indépendants.

```python
from collections.abc import Callable


def fabrique_compteur(pas: int) -> Callable[[], int]:
  valeur = 0

  def compteur() -> int:
    ...
    return valeur

  return compteur
```

### Solution

In [ ]:
from collections.abc import Callable


def fabrique_compteur(pas: int) -> Callable[[], int]:
  valeur = 0

  def compteur() -> int:
    nonlocal valeur
    valeur += pas
    return valeur

  return compteur


unites = fabrique_compteur(1)
dizaines = fabrique_compteur(10)
print(unites(), unites(), dizaines(), unites())

## Fixer l'argument d'une fonction avec une fermeture

Par défaut, la fonction `int` interprète les chaînes de caractères qui lui sont passées comme des nombres en base 10. Écrivez une fonction `fabrique_entier_base(base)` qui renvoie une fermeture : celle-ci convertit une chaîne de caractères en entier dans cette base. Par exemple, avec `entier_base_2 = fabrique_entier_base(2)`, `entier_base_2("101")` vaut 5.

**Documentation utile**

- [`int`](https://docs.python.org/fr/3/library/functions.html#int)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`int` reçoit la base par son argument `base` : `int("101", base=2)` vaut 5. La fermeture ne fait que lire `base`, le paramètre de la fonction englobante.

```python
from collections.abc import Callable


def fabrique_entier_base(base: int) -> Callable[[str], int]:
  def entier_base(x: str) -> int:
    return ...

  return entier_base
```

### Solution

In [ ]:
from collections.abc import Callable


def fabrique_entier_base(base: int) -> Callable[[str], int]:
  def entier_base(x: str) -> int:
    return int(x, base=base)

  return entier_base


entier_base_2 = fabrique_entier_base(2)
print(entier_base_2("101"))
print(entier_base_2.__code__.co_freevars, entier_base_2.__closure__[0].cell_contents)

Pour seulement fixer un argument, la bibliothèque standard fournit [`functools.partial`](https://docs.python.org/fr/3/library/functools.html#functools.partial), sans fermeture à écrire :

In [ ]:
from functools import partial

entier_base_2 = partial(int, base=2)
print(entier_base_2("101"))

## Création de plusieurs fermetures dans une boucle

La liste `multiplications` de la cellule suivante devrait contenir les fonctions qui multiplient par 0, 1, 2, 3 et 4, et la cellule afficher `0 2 4 6 8`. Elle affiche pourtant `8 8 8 8 8`.

Expliquez ce résultat dans un commentaire, puis corrigez la liste de deux façons pour obtenir `0 2 4 6 8` : avec un argument par défaut, puis avec une fonction qui fabrique chaque fermeture.

In [ ]:
multiplications = [lambda x: x * i for i in range(5)]
print(*[m(2) for m in multiplications])

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une correction doit donner à chaque fonction sa propre valeur de `i` :

- un argument par défaut est évalué à la création de la fonction ;
- chaque appel d'une fonction qui reçoit `i` et renvoie la `lambda` crée sa propre variable `i`.

```python
from collections.abc import Callable

multiplications = [lambda x, i=i: ... for i in range(5)]


def fabrique_multiplication(i: int) -> Callable[[int], int]:
  return lambda x: ...


multiplications = [... for i in range(5)]
```

### Solution

Les fonctions capturent la variable `i`, pas sa valeur : elles la lisent à leur appel, quand la boucle est finie et que `i` vaut 4. C'est ce qu'on appelle la liaison tardive (*late binding*). Il faut y faire très attention lorsque l'on crée des fermetures.

In [ ]:
multiplications = [lambda x, i=i: x * i for i in range(5)]
print(*[m(2) for m in multiplications])

In [ ]:
from collections.abc import Callable


def fabrique_multiplication(i: int) -> Callable[[int], int]:
  return lambda x: x * i


multiplications = [fabrique_multiplication(i) for i in range(5)]
print(*[m(2) for m in multiplications])